# Notebook 05 — Player-Match Dataset Construction

## Objective
Construct batting and bowling performance datasets
at player-match granularity from validated IPL deliveries.

## Source tables
- matches.csv
- innings.csv
- deliveries.csv

## Outputs
- batting_player_match.csv
- bowling_player_match.csv

## Validation
- One row per player per match per role.
- No duplicate player-match keys.
- Match identifiers must exist in matches.csv.
- Aggregated statistics must reconcile with source deliveries.

In [22]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR)
print("Directory exists:", PROCESSED_DIR.is_dir())

assert PROCESSED_DIR.is_dir()

Project root: D:\Git\Personal\ipl-player-performance-v2
Processed data directory: D:\Git\Personal\ipl-player-performance-v2\data\processed
Directory exists: True


In [23]:
source_files = {
    "matches": PROCESSED_DIR / "matches.csv",
    "innings": PROCESSED_DIR / "innings.csv",
    "deliveries": PROCESSED_DIR / "deliveries.csv",
}

for name, path in source_files.items():
    print(
        f"{name}:",
        path.exists(),
        f"| Size: {path.stat().st_size / (1024 * 1024):.2f} MB"
        if path.exists() else ""
    )

assert all(path.is_file() for path in source_files.values())
print("\nSOURCE FILE VALIDATION: PASS")

matches: True | Size: 0.34 MB
innings: True | Size: 0.16 MB
deliveries: True | Size: 29.23 MB

SOURCE FILE VALIDATION: PASS


In [24]:
matches = pd.read_csv(
    source_files["matches"],
    dtype={"match_id": "string"},
    low_memory=False
)

innings = pd.read_csv(
    source_files["innings"],
    dtype={"match_id": "string"},
    low_memory=False
)

deliveries = pd.read_csv(
    source_files["deliveries"],
    dtype={"match_id": "string"},
    low_memory=False
)

print("Matches:", matches.shape)
print("Innings:", innings.shape)
print("Deliveries:", deliveries.shape)

Matches: (1243, 26)
Innings: (2514, 14)
Deliveries: (295732, 21)


In [25]:
for name, df in {
    "MATCHES": matches,
    "INNINGS": innings,
    "DELIVERIES": deliveries
}.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Columns:", df.columns.tolist())
    print("\nData types:\n", df.dtypes)
    print("\nFirst 3 records:")
    display(df.head(3))


MATCHES
Columns: ['match_id', 'source_file', 'season', 'date', 'match_type', 'gender', 'team_type', 'overs_limit', 'venue', 'city', 'city_original', 'city_was_imputed', 'team1', 'team2', 'teams_json', 'toss_winner', 'toss_decision', 'winner', 'result', 'method', 'eliminator', 'bowl_out', 'player_of_match', 'event_name', 'event_match_number', 'event_stage']

Data types:
 match_id               string
source_file               str
season                    str
date                      str
match_type                str
gender                    str
team_type                 str
overs_limit             int64
venue                     str
city                      str
city_original             str
city_was_imputed         bool
team1                     str
team2                     str
teams_json                str
toss_winner               str
toss_decision             str
winner                    str
result                    str
method                    str
eliminator                

,match_id,source_file,season,date,match_type,gender,team_type,overs_limit,venue,city,...,toss_decision,winner,result,method,eliminator,bowl_out,player_of_match,event_name,event_match_number,event_stage
0,1082591,1082591.json,2017,2017-04-05,T20,male,club,20,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,...,field,Sunrisers Hyderabad,NaN,NaN,NaN,NaN,"[""Yuvraj Singh""]",Indian Premier League,1.0,NaN
1,1082592,1082592.json,2017,2017-04-06,T20,male,club,20,Maharashtra Cricket Association Stadium,Pune,...,field,Rising Pune Supergiant,NaN,NaN,NaN,NaN,"[""SPD Smith""]",Indian Premier League,2.0,NaN
2,1082593,1082593.json,2017,2017-04-07,T20,male,club,20,Saurashtra Cricket Association Stadium,Rajkot,...,field,Kolkata Knight Riders,NaN,NaN,NaN,NaN,"[""CA Lynn""]",Indian Premier League,3.0,NaN



INNINGS
Columns: ['match_id', 'innings_number', 'batting_team', 'is_super_over', 'target_runs', 'target_overs', 'overs_recorded', 'delivery_count', 'legal_deliveries', 'total_runs', 'extras_runs', 'batter_runs', 'wicket_records', 'actual_dismissals']

Data types:
 match_id              string
innings_number         int64
batting_team             str
is_super_over           bool
target_runs          float64
target_overs         float64
overs_recorded         int64
delivery_count         int64
legal_deliveries       int64
total_runs             int64
extras_runs            int64
batter_runs            int64
wicket_records         int64
actual_dismissals      int64
dtype: object

First 3 records:


,match_id,innings_number,batting_team,is_super_over,target_runs,target_overs,overs_recorded,delivery_count,legal_deliveries,total_runs,extras_runs,batter_runs,wicket_records,actual_dismissals
0,1082591,1,Sunrisers Hyderabad,False,NaN,NaN,20,125,120,207,7,200,4,4
1,1082591,2,Royal Challengers Bangalore,False,208.0,20.0,20,123,118,172,6,166,10,10
2,1082592,1,Mumbai Indians,False,NaN,NaN,20,125,120,184,8,176,8,8



DELIVERIES
Columns: ['match_id', 'innings_number', 'over_number', 'delivery_index_in_over', 'batting_team', 'batter', 'bowler', 'non_striker', 'batter_runs', 'extras_runs', 'total_runs', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'is_legal_delivery', 'wicket_count', 'wicket_kinds', 'wickets_json', 'extras_json']

Data types:
 match_id                  string
innings_number             int64
over_number                int64
delivery_index_in_over     int64
batting_team                 str
batter                       str
bowler                       str
non_striker                  str
batter_runs                int64
extras_runs                int64
total_runs                 int64
wides                      int64
noballs                    int64
byes                       int64
legbyes                    int64
penalty                    int64
is_legal_delivery           bool
wicket_count               int64
wicket_kinds                 str
wickets_json                 str
extr

,match_id,innings_number,over_number,delivery_index_in_over,batting_team,batter,bowler,non_striker,batter_runs,extras_runs,...,wides,noballs,byes,legbyes,penalty,is_legal_delivery,wicket_count,wicket_kinds,wickets_json,extras_json
0,1082591,1,0,1,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0,0,...,0,0,0,0,0,True,0,[],[],{}
1,1082591,1,0,2,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0,0,...,0,0,0,0,0,True,0,[],[],{}
2,1082591,1,0,3,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,4,0,...,0,0,0,0,0,True,0,[],[],{}


In [26]:

required_columns = {
    "matches": {
        "match_id", "season", "date", "venue", "city"
    },
    "innings": {
        "match_id", "innings_number", "batting_team", "is_super_over"
    },
    "deliveries": {
        "match_id", "innings_number", "batting_team",
        "batter", "bowler", "non_striker",
        "batter_runs", "extras_runs", "total_runs",
        "wides", "noballs", "byes", "legbyes", "penalty",
        "is_legal_delivery", "wicket_count",
        "wicket_kinds", "wickets_json"
    }
}

dataframes = {
    "matches": matches,
    "innings": innings,
    "deliveries": deliveries
}

for name, required in required_columns.items():
    missing = required - set(dataframes[name].columns)
    print(f"{name}: missing required columns = {sorted(missing)}")
    assert not missing, f"Required columns missing from {name}"

print("\nSOURCE SCHEMA VALIDATION: PASS")

matches: missing required columns = []
innings: missing required columns = []
deliveries: missing required columns = []

SOURCE SCHEMA VALIDATION: PASS


In [27]:
expected_rows = {
    "matches": 1243,
    "innings": 2514,
    "deliveries": 295732
}

for name, df in dataframes.items():
    print(f"{name}: {len(df):,} rows")
    assert len(df) == expected_rows[name], (
        f"Unexpected row count for {name}"
    )

assert matches["match_id"].is_unique

assert not innings.duplicated(
    ["match_id", "innings_number"]
).any()

assert not deliveries.duplicated(
    [
        "match_id",
        "innings_number",
        "over_number",
        "delivery_index_in_over"
    ]
).any()

print("\nSOURCE DATA INTEGRITY: PASS")

matches: 1,243 rows
innings: 2,514 rows
deliveries: 295,732 rows

SOURCE DATA INTEGRITY: PASS


In [28]:
print("DELIVERY COLUMNS:")
print(deliveries.columns.tolist())

print("\nWICKET JSON SAMPLE:")
wicket_samples = deliveries.loc[
    deliveries["wickets_json"].notna()
    & deliveries["wickets_json"].astype(str).ne("[]"),
    ["match_id", "innings_number", "batter", "bowler",
     "wicket_count", "wicket_kinds", "wickets_json"]
].head(5)

display(wicket_samples)

print("\nWICKET JSON DATA TYPE:")
print(deliveries["wickets_json"].dtype)

print("\nSUPER-OVER COUNTS:")
print(innings["is_super_over"].value_counts(dropna=False))

DELIVERY COLUMNS:
['match_id', 'innings_number', 'over_number', 'delivery_index_in_over', 'batting_team', 'batter', 'bowler', 'non_striker', 'batter_runs', 'extras_runs', 'total_runs', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'is_legal_delivery', 'wicket_count', 'wicket_kinds', 'wickets_json', 'extras_json']

WICKET JSON SAMPLE:


,match_id,innings_number,batter,bowler,wicket_count,wicket_kinds,wickets_json
11,1082591,1,DA Warner,A Choudhary,1,"[""caught""]","[{""kind"": ""caught"", ""player_out"": ""DA Warner"",..."
64,1082591,1,S Dhawan,STR Binny,1,"[""caught""]","[{""kind"": ""caught"", ""player_out"": ""S Dhawan"", ..."
94,1082591,1,MC Henriques,YS Chahal,1,"[""caught""]","[{""kind"": ""caught"", ""player_out"": ""MC Henrique..."
116,1082591,1,Yuvraj Singh,TS Mills,1,"[""bowled""]","[{""kind"": ""bowled"", ""player_out"": ""Yuvraj Sing..."
160,1082591,2,Mandeep Singh,Rashid Khan,1,"[""bowled""]","[{""kind"": ""bowled"", ""player_out"": ""Mandeep Sin..."



WICKET JSON DATA TYPE:
str

SUPER-OVER COUNTS:
is_super_over
False    2480
True       34
Name: count, dtype: int64


In [29]:
import json

wicket_rows = deliveries.loc[
    deliveries["wickets_json"].notna()
    & deliveries["wickets_json"].astype(str).ne("[]"),
    ["match_id", "innings_number", "batter", "bowler",
     "wicket_count", "wicket_kinds", "wickets_json"]
].copy()

print("Total delivery rows containing wicket records:", len(wicket_rows))

for idx, row in wicket_rows.head(10).iterrows():
    print(f"\nMatch: {row['match_id']}")
    print(f"Innings: {row['innings_number']}")
    print(f"Batter: {row['batter']}")
    print(f"Bowler: {row['bowler']}")
    print("Wicket JSON:", row["wickets_json"])
    print("Parsed:", json.loads(row["wickets_json"]))

print("\nALL DELIVERY COLUMNS:")
for col in deliveries.columns:
    print(col)

Total delivery rows containing wicket records: 14705

Match: 1082591
Innings: 1
Batter: DA Warner
Bowler: A Choudhary
Wicket JSON: [{"kind": "caught", "player_out": "DA Warner", "fielders": [{"name": "Mandeep Singh"}]}]
Parsed: [{'kind': 'caught', 'player_out': 'DA Warner', 'fielders': [{'name': 'Mandeep Singh'}]}]

Match: 1082591
Innings: 1
Batter: S Dhawan
Bowler: STR Binny
Wicket JSON: [{"kind": "caught", "player_out": "S Dhawan", "fielders": [{"name": "Sachin Baby"}]}]
Parsed: [{'kind': 'caught', 'player_out': 'S Dhawan', 'fielders': [{'name': 'Sachin Baby'}]}]

Match: 1082591
Innings: 1
Batter: MC Henriques
Bowler: YS Chahal
Wicket JSON: [{"kind": "caught", "player_out": "MC Henriques", "fielders": [{"name": "Sachin Baby"}]}]
Parsed: [{'kind': 'caught', 'player_out': 'MC Henriques', 'fielders': [{'name': 'Sachin Baby'}]}]

Match: 1082591
Innings: 1
Batter: Yuvraj Singh
Bowler: TS Mills
Wicket JSON: [{"kind": "bowled", "player_out": "Yuvraj Singh"}]
Parsed: [{'kind': 'bowled', 'pla

In [30]:
# Parse all wicket records
all_wickets = []

for value in deliveries["wickets_json"].dropna():
    records = json.loads(value)

    if isinstance(records, list):
        all_wickets.extend(records)

wickets_df = pd.DataFrame(all_wickets)

print("Total parsed wicket records:", len(wickets_df))

print("\nDismissal kinds and counts:")
display(
    wickets_df["kind"]
    .value_counts(dropna=False)
    .rename_axis("dismissal_kind")
    .reset_index(name="count")
)

print("\nMissing player_out values:")
print(wickets_df["player_out"].isna().sum())

print("\nUnique dismissal kinds:")
print(sorted(wickets_df["kind"].dropna().unique()))

Total parsed wicket records: 14705

Dismissal kinds and counts:


,dismissal_kind,count
0,caught,9321
1,bowled,2460
2,run out,1194
3,lbw,883
4,caught and bowled,410
5,stumped,388
6,hit wicket,20
7,retired hurt,19
8,retired out,6
9,obstructing the field,4



Missing player_out values:
0

Unique dismissal kinds:
['bowled', 'caught', 'caught and bowled', 'hit wicket', 'lbw', 'obstructing the field', 'retired hurt', 'retired out', 'run out', 'stumped']


In [31]:

print("EXTRAS JSON SAMPLES:")

extras_samples = deliveries.loc[
    deliveries["extras_json"].notna()
    & deliveries["extras_json"].astype(str).ne("{}"),
    ["extras_runs", "wides", "noballs", "byes",
     "legbyes", "penalty", "extras_json"]
].head(10)

display(extras_samples)

EXTRAS JSON SAMPLES:


,extras_runs,wides,noballs,byes,legbyes,penalty,extras_json
4,2,2,0,0,0,0,"{""wides"": 2}"
6,1,0,0,0,1,0,"{""legbyes"": 1}"
9,1,0,1,0,0,0,"{""noballs"": 1}"
78,1,1,0,0,0,0,"{""wides"": 1}"
107,1,1,0,0,0,0,"{""wides"": 1}"
108,1,1,0,0,0,0,"{""wides"": 1}"
146,1,1,0,0,0,0,"{""wides"": 1}"
153,1,1,0,0,0,0,"{""wides"": 1}"
176,1,1,0,0,0,0,"{""wides"": 1}"
182,1,0,1,0,0,0,"{""noballs"": 1}"


In [32]:

# Cricket-statistics definitions

BOWLER_CREDITED_DISMISSALS = {
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "stumped",
    "hit wicket",
}

# Retired hurt is not an official dismissal.
NON_DISMISSAL_KINDS = {"retired hurt"}

OUTPUT_DIR = PROCESSED_DIR / "player_match"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Bowler-creditable dismissal types:",
      sorted(BOWLER_CREDITED_DISMISSALS))

Output directory: D:\Git\Personal\ipl-player-performance-v2\data\processed\player_match
Bowler-creditable dismissal types: ['bowled', 'caught', 'caught and bowled', 'hit wicket', 'lbw', 'stumped']


In [33]:
# Join innings metadata to every delivery.
delivery_data = deliveries.merge(
    innings[
        ["match_id", "innings_number", "is_super_over"]
    ],
    on=["match_id", "innings_number"],
    how="left",
    validate="many_to_one"
)

assert delivery_data["is_super_over"].notna().all()

# Convert cricket-statistics columns to numeric.
numeric_columns = [
    "batter_runs", "extras_runs", "total_runs",
    "wides", "noballs", "byes", "legbyes",
    "penalty", "wicket_count"
]

for col in numeric_columns:
    delivery_data[col] = pd.to_numeric(
        delivery_data[col], errors="coerce"
    ).fillna(0)

# Normalize the legal-delivery flag.
delivery_data["is_legal_delivery"] = (
    delivery_data["is_legal_delivery"]
    .astype(str)
    .str.lower()
    .map({"true": True, "false": False, "1": True, "0": False})
)

assert delivery_data["is_legal_delivery"].notna().all()

# Balls faced: exclude wides, include no-balls.
delivery_data["balls_faced"] = (
    delivery_data["wides"].eq(0).astype(int)
)

# Bowler's runs conceded: batter runs + wides + no-balls.
delivery_data["bowler_runs_conceded"] = (
    delivery_data["batter_runs"]
    + delivery_data["wides"]
    + delivery_data["noballs"]
)

print("Delivery rows:", len(delivery_data))
print("Super-over delivery rows:",
      int(delivery_data["is_super_over"].sum()))
print("Regular delivery rows:",
      int((~delivery_data["is_super_over"]).sum()))

Delivery rows: 295732
Super-over delivery rows: 175
Regular delivery rows: 295557


In [34]:
import json

wicket_records = []

for row in delivery_data[
    [
        "match_id", "innings_number", "is_super_over",
        "batting_team", "bowler", "wickets_json"
    ]
].itertuples(index=False):

    raw_json = row.wickets_json

    if pd.isna(raw_json) or str(raw_json).strip() in ("", "[]"):
        continue

    records = json.loads(raw_json)

    for wicket in records:
        wicket_records.append({
            "match_id": row.match_id,
            "innings_number": row.innings_number,
            "is_super_over": row.is_super_over,
            "batting_team": row.batting_team,
            "bowler": row.bowler,
            "player_out": wicket.get("player_out"),
            "dismissal_kind": wicket.get("kind")
        })

dismissals = pd.DataFrame(wicket_records)

assert len(dismissals) == 14705
assert dismissals["player_out"].notna().all()

dismissals["is_actual_dismissal"] = (
    ~dismissals["dismissal_kind"].isin(NON_DISMISSAL_KINDS)
)

dismissals["is_bowler_wicket"] = (
    dismissals["dismissal_kind"].isin(
        BOWLER_CREDITED_DISMISSALS
    )
)

print("Parsed dismissal records:", len(dismissals))
print("\nDismissal counts:")
display(dismissals["dismissal_kind"].value_counts())

Parsed dismissal records: 14705

Dismissal counts:


dismissal_kind
caught                   9321
bowled                   2460
run out                  1194
lbw                       883
caught and bowled         410
stumped                   388
hit wicket                 20
retired hurt               19
retired out                 6
obstructing the field       4
Name: count, dtype: int64

In [35]:
# Aggregate delivery-based batting statistics.
batting_base = (
    delivery_data
    .dropna(subset=["batter"])
    .groupby(
        ["match_id", "batter", "is_super_over", "batting_team"],
        as_index=False
    )
    .agg(
        runs_scored=("batter_runs", "sum"),
        balls_faced=("balls_faced", "sum"),
        fours=("batter_runs", lambda s: (s == 4).sum()),
        sixes=("batter_runs", lambda s: (s == 6).sum()),
        innings_batted=("innings_number", "nunique")
    )
)

# Add players dismissed without facing a delivery.
dismissal_batters = (
    dismissals[
        ["match_id", "player_out", "is_super_over", "batting_team"]
    ]
    .drop_duplicates()
    .rename(columns={"player_out": "batter"})
)

batting_keys = pd.concat(
    [
        batting_base[
            ["match_id", "batter", "is_super_over", "batting_team"]
        ],
        dismissal_batters
    ],
    ignore_index=True
).drop_duplicates()

batting = batting_keys.merge(
    batting_base,
    on=["match_id", "batter", "is_super_over", "batting_team"],
    how="left",
    validate="one_to_one"
)

batting_stats = (
    dismissals
    .groupby(
        ["match_id", "player_out", "is_super_over"],
        as_index=False
    )
    .agg(
        dismissals=("is_actual_dismissal", "sum"),
        retired_hurt_count=(
            "dismissal_kind",
            lambda s: (s == "retired hurt").sum()
        )
    )
    .rename(columns={"player_out": "batter"})
)

batting = batting.merge(
    batting_stats,
    on=["match_id", "batter", "is_super_over"],
    how="left",
    validate="one_to_one"
)

batting["runs_scored"] = batting["runs_scored"].fillna(0).astype(int)
batting["balls_faced"] = batting["balls_faced"].fillna(0).astype(int)
batting["fours"] = batting["fours"].fillna(0).astype(int)
batting["sixes"] = batting["sixes"].fillna(0).astype(int)
batting["innings_batted"] = batting["innings_batted"].fillna(0).astype(int)
batting["dismissals"] = batting["dismissals"].fillna(0).astype(int)
batting["retired_hurt_count"] = (
    batting["retired_hurt_count"].fillna(0).astype(int)
)

batting["not_out"] = batting["dismissals"].eq(0).astype(int)

batting["strike_rate"] = np.where(
    batting["balls_faced"] > 0,
    batting["runs_scored"] / batting["balls_faced"] * 100,
    np.nan
)

batting["batting_average"] = np.where(
    batting["dismissals"] > 0,
    batting["runs_scored"] / batting["dismissals"],
    np.nan
)

batting = batting.rename(columns={"batter": "player"})

print("Batting player-match rows:", len(batting))
display(batting.head())

Batting player-match rows: 18897


,match_id,player,is_super_over,batting_team,runs_scored,balls_faced,fours,sixes,innings_batted,dismissals,retired_hurt_count,not_out,strike_rate,batting_average
0,1082591,A Choudhary,False,Royal Challengers Bangalore,6,2,0,1,1,0,0,1,300.000000,NaN
1,1082591,BCJ Cutting,False,Sunrisers Hyderabad,16,6,0,2,1,0,0,1,266.666667,NaN
2,1082591,CH Gayle,False,Royal Challengers Bangalore,32,21,2,3,1,1,0,0,152.380952,32.0
3,1082591,DA Warner,False,Sunrisers Hyderabad,14,8,2,1,1,1,0,0,175.000000,14.0
4,1082591,DJ Hooda,False,Sunrisers Hyderabad,16,12,0,1,1,0,0,1,133.333333,NaN


In [36]:
# Build the complete player-innings population.
# Include batters who faced deliveries and batters dismissed without facing.

delivery_batting_innings = delivery_data[
    ["match_id", "innings_number", "is_super_over",
     "batting_team", "batter"]
].rename(columns={"batter": "player"})

dismissal_batting_innings = dismissals[
    ["match_id", "innings_number", "is_super_over",
     "batting_team", "player_out"]
].rename(columns={"player_out": "player"})

all_batting_innings = pd.concat(
    [delivery_batting_innings, dismissal_batting_innings],
    ignore_index=True
).drop_duplicates()

innings_counts = (
    all_batting_innings
    .groupby(
        ["match_id", "player", "is_super_over", "batting_team"],
        as_index=False
    )
    .agg(innings_batted=("innings_number", "nunique"))
)

# Replace the old innings_batted calculation.
batting = batting.drop(columns=["innings_batted"]).merge(
    innings_counts,
    on=["match_id", "player", "is_super_over", "batting_team"],
    how="left",
    validate="one_to_one"
)

# Count innings in which the player was not dismissed.
# Retired hurt is not counted as an actual dismissal.
batting["not_out_innings"] = (
    batting["innings_batted"] - batting["dismissals"]
).clip(lower=0)

# Remove the ambiguous match-level flag.
batting = batting.drop(columns=["not_out"], errors="ignore")

assert batting["innings_batted"].ge(1).all()
assert batting["not_out_innings"].le(
    batting["innings_batted"]
).all()

print("Batting innings metrics corrected.")
print("Total batting innings:", batting["innings_batted"].sum())
print("Total not-out innings:", batting["not_out_innings"].sum())

Batting innings metrics corrected.
Total batting innings: 18897
Total not-out innings: 4211


In [37]:
bowling_base = (
    delivery_data
    .dropna(subset=["bowler"])
    .groupby(
        ["match_id", "bowler", "is_super_over"],
        as_index=False
    )
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("is_legal_delivery", "sum"),
        wides_conceded=("wides", "sum"),
        noballs_conceded=("noballs", "sum"),
        innings_bowled=("innings_number", "nunique")
    )
)

bowling_wickets = (
    dismissals
    .groupby(
        ["match_id", "bowler", "is_super_over"],
        as_index=False
    )
    .agg(
        wickets=("is_bowler_wicket", "sum")
    )
)

bowling = bowling_base.merge(
    bowling_wickets,
    on=["match_id", "bowler", "is_super_over"],
    how="left",
    validate="one_to_one"
)

bowling["wickets"] = bowling["wickets"].fillna(0).astype(int)

bowling["overs_bowled"] = bowling["legal_balls"] / 6

bowling["economy_rate"] = np.where(
    bowling["legal_balls"] > 0,
    bowling["runs_conceded"] / (bowling["legal_balls"] / 6),
    np.nan
)

bowling["bowling_average"] = np.where(
    bowling["wickets"] > 0,
    bowling["runs_conceded"] / bowling["wickets"],
    np.nan
)

bowling["bowling_strike_rate"] = np.where(
    bowling["wickets"] > 0,
    bowling["legal_balls"] / bowling["wickets"],
    np.nan
)

bowling = bowling.rename(columns={"bowler": "player"})

print("Bowling player-match rows:", len(bowling))
display(bowling.head())

Bowling player-match rows: 14734


,match_id,player,is_super_over,runs_conceded,legal_balls,wides_conceded,noballs_conceded,innings_bowled,wickets,overs_bowled,economy_rate,bowling_average,bowling_strike_rate
0,1082591,A Choudhary,False,55,24,3,1,1,1,4.000000,13.750000,55.0,24.0
1,1082591,A Nehra,False,42,24,1,0,1,2,4.000000,10.500000,21.0,12.0
2,1082591,B Kumar,False,27,24,1,0,1,2,4.000000,6.750000,13.5,12.0
3,1082591,BCJ Cutting,False,35,22,2,0,1,0,3.666667,9.545455,NaN,NaN
4,1082591,Bipul Sharma,False,4,6,0,0,1,1,1.000000,4.000000,4.0,6.0


In [38]:
match_metadata = matches[
    ["match_id", "season", "date", "match_type"]
].copy()

batting = batting.merge(
    match_metadata,
    on="match_id",
    how="left",
    validate="many_to_one"
)

bowling = bowling.merge(
    match_metadata,
    on="match_id",
    how="left",
    validate="many_to_one"
)

assert batting["season"].notna().all()
assert bowling["season"].notna().all()

# Save regular-innings and super-over datasets separately.
batting_regular = batting.loc[~batting["is_super_over"]].copy()
batting_super = batting.loc[batting["is_super_over"]].copy()

bowling_regular = bowling.loc[~bowling["is_super_over"]].copy()
bowling_super = bowling.loc[bowling["is_super_over"]].copy()

outputs = {
    "batting_player_match_regular.csv": batting_regular,
    "batting_player_match_super_over.csv": batting_super,
    "bowling_player_match_regular.csv": bowling_regular,
    "bowling_player_match_super_over.csv": bowling_super,
}

for filename, df in outputs.items():
    df.to_csv(OUTPUT_DIR / filename, index=False)
    print(f"Saved {filename}: {len(df):,} rows")

print("\nOutput directory:", OUTPUT_DIR)

Saved batting_player_match_regular.csv: 18,822 rows
Saved batting_player_match_super_over.csv: 75 rows
Saved bowling_player_match_regular.csv: 14,700 rows
Saved bowling_player_match_super_over.csv: 34 rows

Output directory: D:\Git\Personal\ipl-player-performance-v2\data\processed\player_match


In [39]:
# Unique player-match-role keys.
assert not batting.duplicated(
    ["match_id", "player", "is_super_over"]
).any()

assert not bowling.duplicated(
    ["match_id", "player", "is_super_over"]
).any()

# No missing player names.
assert batting["player"].notna().all()
assert bowling["player"].notna().all()

# Batter runs must reconcile with source delivery batter runs.
assert batting["runs_scored"].sum() == delivery_data["batter_runs"].sum()

# Bowler wickets must reconcile with the parsed bowler-creditable dismissals.
assert bowling["wickets"].sum() == dismissals["is_bowler_wicket"].sum()

# Regular + super-over rows must equal combined datasets.
assert len(batting_regular) + len(batting_super) == len(batting)
assert len(bowling_regular) + len(bowling_super) == len(bowling)

print("Batting rows:", len(batting))
print("Bowling rows:", len(bowling))
print("Batting runs reconciled:", batting["runs_scored"].sum())
print("Bowler wickets reconciled:", bowling["wickets"].sum())
print("Dismissals in source:", dismissals["is_actual_dismissal"].sum())
print("\nPLAYER-MATCH AGGREGATION VALIDATION: PASS")

Batting rows: 18897
Bowling rows: 14734
Batting runs reconciled: 381529
Bowler wickets reconciled: 13482
Dismissals in source: 14686

PLAYER-MATCH AGGREGATION VALIDATION: PASS
